# Baseline screening — MLP & XGBoost on M2OR (molecule ‖ protein)

Runs the two MP baselines across **splits** and **protein variants**, with on-disk
caching (`notebooks/cache/results.json`) so results survive kernel restarts and
already-computed configs are never recomputed.

The last section is a placeholder to plug in **modified ESM-2** embeddings as new
protein variants — only the new variant gets computed, the rest is reused.

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# locate repo root (has pyproject.toml) regardless of where the kernel started
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from orbind import dataset as D
from orbind.baselines import run_one

DATA = ROOT / 'data'
CACHE = ROOT / 'notebooks' / 'cache'; CACHE.mkdir(parents=True, exist_ok=True)
print('repo root:', ROOT)

In [ ]:
# --- load data + embeddings -------------------------------------------------
PAIRS = pd.read_csv(DATA / 'processed' / 'pairs_curated.csv')
ESM = D.load_npz_dict(DATA / 'embeddings' / 'proteins' / 'esm2_650m.npz')
GIN = D.load_npz_dict(DATA / 'embeddings' / 'molecules' / 'gin_supervised_contextpred.npz')
print(f'pairs={len(PAIRS)} | proteins={len(ESM)} | molecules={len(GIN)}')

# protein variants: tag -> (protein_dict, random_prot_flag)
PROT_VARIANTS = {
    'ESM':    (ESM, False),   # real ESM-2 650M
    'random': (ESM, True),    # per-row noise floor (control)
    # next step: add modified ESM-2 here, e.g. 'ESM_pca': (esm_pca, False)
}

In [ ]:
# --- on-disk result cache ---------------------------------------------------
CACHE_FILE = CACHE / 'results.json'

def _load(): return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
def _save(c): CACHE_FILE.write_text(json.dumps(c, indent=2))

def cached_run(tag, prot, random_prot, head, split, seed=42, force=False):
    cache = _load()
    key = f'{tag}|{head}|{split}|seed{seed}'
    if key in cache and not force:
        return cache[key]
    m, info = run_one(PAIRS, prot, GIN, head, split, random_prot=random_prot, seed=seed)
    rec = {'tag': tag, 'head': head, 'split': split, 'seed': seed,
           **{k: round(float(v), 4) for k, v in m.items()}, **info}
    cache[key] = rec; _save(cache)
    return rec

In [ ]:
# --- run the grid (cached) --------------------------------------------------
HEADS  = ['mlp', 'boost']
SPLITS = ['stratified', 'group_molecule', 'group_receptor']

rows = []
for tag, (prot, rnd) in PROT_VARIANTS.items():
    for head in HEADS:
        for split in SPLITS:
            r = cached_run(tag, prot, rnd, head, split)
            rows.append(r)
            print(f"{tag:8s} {head:5s} {split:16s} AUPRC={r['AUPRC']:.3f} AUROC={r['AUROC']:.3f}")
res = pd.DataFrame(rows)

In [ ]:
# --- view: per head, metric x (variant, split) ------------------------------
for head in HEADS:
    sub = res[res['head'] == head]
    piv = sub.pivot_table(index='split', columns='tag', values=['AUPRC', 'AUROC', 'MCC'])
    print(f'
### {head}')
    display(piv.round(3))

## Next step — modified ESM-2 embeddings

Add a transformed copy of `ESM` as a new variant, then re-run the grid cell. Only
the new variant is computed; cached configs are reused.

```python
def transform_esm(esm):
    # e.g. PCA / scaling / dim masking / noise injection / projection
    return {k: v for k, v in esm.items()}  # <- replace with your transform

PROT_VARIANTS['ESM_mod'] = (transform_esm(ESM), False)
# re-run the grid cell; pass force=True to cached_run to overwrite an entry
```